In [ ]:
#| default_exp main

In [ ]:
#| export
import os
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from dotenv import load_dotenv
from fasthtml.common import *
from monsterui.all import *
from lisette import *
from asyncio import sleep
import httpx
import cole_compass
from cole_compass.table_and_charts import (
    ingest_parsed, table_view, table_panel, table_badge, register_table_routes,
    chart_view, chart_panel, chart_badge, analyze_reply,
    get_latest_table, get_analyst_table_id, set_analyst_table_id,
    df_records_for_analyst, store_chart_from_plotly_json,
)

STATIC_DIR = Path(cole_compass.__file__).parent

load_dotenv()
_inference_url = os.environ["INFERENCE_CONFIG_1_URL"].removesuffix("/chat/completions")
_inference_model = os.environ["INFERENCE_CONFIG_1_MODEL"]
CONTROLLER_URL = os.environ["CONTROLLER_URL"]
DATA_ANALYST_URL = (os.environ.get("DATA_ANALYST_URL") or "http://127.0.0.1:5030").rstrip("/")
chat = AsyncChat(model=f"openai/{_inference_model}", api_base=_inference_url, api_key="not-needed")

## Modes

* **Chat** (default) — messages go to blue_v02 controller
* **Table** — if a table is loaded, messages go to the data analyst agent; otherwise a hint is shown
* Active mode is a hidden form field updated when taskbar icons are clicked

In [ ]:
#| export
class IconKind(Enum):
    INTERFACE = "interface"   # black icons - open chat, sections, etc.
    VISUAL    = "visual"      # blue icons  - data visualizations

@dataclass
class TaskbarIcon:
    id: str
    label: str
    icon: str
    kind: IconKind
    active: bool = False  # outline shown only when this icon is the selected one
    extra: dict = field(default_factory=dict)

# ---sample icons ----
interface_icons = [
    TaskbarIcon("chat", "Chat", "message-circle", IconKind.INTERFACE, active=True),
]
visual_icons = [
    TaskbarIcon("chart1", "Chart", "bar-chart-2", IconKind.VISUAL),
    TaskbarIcon("table1", "Table", "table-2", IconKind.VISUAL),
]

_VIEW_IDS = ("chat-view", "table-view", "chart-view")
_ACTIVE_OUTLINE_CLASSES = ("ring-2", "ring-primary")

def _show_view_js(view_id: str, *, extra: str = "") -> str:
    parts = [f"document.getElementById('{v}').classList.add('hidden')" for v in _VIEW_IDS]
    parts.append(f"document.getElementById('{view_id}').classList.remove('hidden')")
    if extra:
        parts.append(extra)
    return "; ".join(parts)

def _set_active_icon_js(icon_id: str) -> str:
    """Client-side: outline only the clicked taskbar icon."""
    remove = "".join(
        f"b.classList.remove('{c}');" for c in _ACTIVE_OUTLINE_CLASSES
    )
    add = "".join(
        f"t.classList.add('{c}');" for c in _ACTIVE_OUTLINE_CLASSES
    )
    return (
        "document.querySelectorAll('[data-taskbar-icon]').forEach(b=>{"
        + remove
        + "});"
        + f"const t=document.querySelector('[data-taskbar-icon=\"{icon_id}\"]');"
        + f"if(t){{{add}}}"
    )


def _set_ui_mode_js(mode: str) -> str:
    """Update the hidden form field so /chat/send knows chat vs table routing."""
    ph = "Ask about this table…" if mode == "table" else "Type a message…"
    return (
        "const m=document.getElementById('ui-mode');"
        f"if(m)m.value={mode!r};"
        "const i=document.getElementById('chat-input');"
        f"if(i)i.placeholder={ph!r};"
    )


def _icon_btn(ic: TaskbarIcon, *, color: str = "", **kw):
    outline = " ".join(_ACTIVE_OUTLINE_CLASSES) if ic.active else ""
    return Button(
        Div(
            UkIcon(ic.icon, cls="w-4 h-4"),
            Span(ic.label, cls="text-[10px] leading-tight"),
            cls="flex flex-col items-center gap-0.5",
        ),
        title=ic.label,
        id=f"taskbar-icon-{ic.id}",
        data_taskbar_icon=ic.id,
        cls=f"btn btn-ghost btn-sm h-auto py-1 px-2 {color} {outline}".strip(),
        **kw,
    )

def render_icon(ic: TaskbarIcon):
    color = "text-primary" if ic.kind == IconKind.VISUAL else ""
    if ic.id == "chat":
        return _icon_btn(
            ic, color=color,
            onclick="; ".join([
                _set_active_icon_js(ic.id),
                _set_ui_mode_js("chat"),
                _show_view_js("chat-view"),
            ]),
        )
    if ic.id == "table1":
        return Div(
            _icon_btn(
                ic, color=color,
                hx_get="/view/table", hx_target="#table-panel", hx_swap="outerHTML",
                hx_on_click="; ".join([
                    _set_active_icon_js(ic.id),
                    _set_ui_mode_js("table"),
                    _show_view_js(
                        "table-view",
                        extra="requestAnimationFrame(()=>{const el=document.querySelector('#table-panel .js-plotly-plot');if(el&&window.Plotly)Plotly.Plots.resize(el);})",
                    ),
                ]),
            ),
            table_badge(),
            cls="relative",
        )
    if ic.id == "chart1":
        return Div(
            _icon_btn(
                ic, color=color,
                hx_get="/view/chart", hx_target="#chart-panel", hx_swap="outerHTML",
                hx_on_click="; ".join([
                    _set_active_icon_js(ic.id),
                    _set_ui_mode_js("table"),  # chart view still analyzes the loaded table
                    _show_view_js(
                        "chart-view",
                        extra="requestAnimationFrame(()=>{const el=document.querySelector('#chart-panel .js-plotly-plot');if(el&&window.Plotly)Plotly.Plots.resize(el);})",
                    ),
                ]),
            ),
            chart_badge(),
            cls="relative",
        )
    return _icon_btn(
        ic, color=color,
        onclick=_set_active_icon_js(ic.id),
    )

def taskbar():
        return Div(
            *[render_icon(i) for i in interface_icons],
            Div(cls="divider divider-horizontal mx-1"),
            *[render_icon(i) for i in visual_icons],
            Div(cls="flex-1"),    # spacer
            Button(
                Div(
                    UkIcon("grid-2x2", cls="w-4 h-4"),
                    Span("All elements", cls="text-[10px] leading-tight"),
                    cls="flex flex-col items-center gap-0.5",
                ),
                title="All elements",
                cls="btn btn-sm  h-auto py-1 px-2",
            ),
            cls = "inline-flex items-center gap-1 p-2 border-b bg-base-200"
    )

def chat_bubble(role: str, content: str):
    is_user = role == "user"
    return Div(
        Div(content,
            cls=f"chat-bubble whitespace-pre-line {'chat-bubble-primary' if is_user else 'chat-bubble-neutral'}"),
        cls=f"chat {'chat-end' if is_user else 'chat-start'} px-4"
    )

def oob_chat_bubble(role: str, content: str):
    """Append a chat bubble via HTMX OOB (wrapper stripped by beforeend)."""
    return Div(
        chat_bubble(role, content),
        hx_swap_oob="beforeend:#chat-messages",
    )

def chat_area():
    """ Empty scrollable container; bubbles are appended by HTMX"""
    return Div(
        Div(id="chat-messages", cls="flex flex-col gap-2 py-4 w-full max-w-2xl mx-auto"),
        id="chat-view",
        cls="flex flex-col flex-1 overflow-y-auto",
    )

def input_bar():
    return Div(
        Form(
            Input(type="hidden", name="ui_mode", id="ui-mode", value="chat"),
            Div(
                Textarea(
                    placeholder="Type a message…",
                    name="message",
                    rows=2,
                    cls="textarea textarea-bordered w-full resize-none pr-14 focus:outline-none",
                    id="chat-input",
                    onkeydown="if(event.key==='Enter'&&!event.shiftKey){event.preventDefault();this.form.requestSubmit()}",
                    hx_on_input="""
                        let btn = document.getElementById('submit-btn');
                        if (this.value.trim().length > 0) {
                            btn.classList.remove('hidden');
                        } else {
                            btn.classList.add('hidden');
                        }
                        """
                ),
                Button(UkIcon("arrow-up", cls="w-5 h-5"),
                       id="submit-btn",
                       cls="btn btn-primary btn-circle btn-sm absolute right-3 bottom-3 hidden"),
                cls="relative w-full",
            ),
            hx_post="/chat/send",
            hx_target="#htmx-sink",
            hx_swap="none",
            hx_on__after_request="document.getElementById('chat-input').value=''; document.getElementById('submit-btn').classList.add('hidden')",
            cls="w-full max-w-2xl mx-auto",
        ),
        cls="p-3 border-t bg-base-100"
    )

app, rt = fast_app(
    hdrs=Theme.blue.headers(),
    static_path=str(STATIC_DIR),
)
register_table_routes(rt)

@rt("/")
def get(session):
    if 'session_id' not in session:
        import uuid
        session['session_id'] = str(uuid.uuid4())

    return Div(
        Div(taskbar(), cls="flex justify-center border-b"),
        Div(
            chat_area(),
            table_view(),
            chart_view(),
            cls="flex flex-col flex-1 min-h-0 overflow-hidden",
        ),
        input_bar(),
        Div(id="htmx-sink", cls="hidden", aria_hidden="true"),
        cls="flex flex-col h-screen",
    )


async def _ensure_analyst_table(client: httpx.AsyncClient, table) -> str:
    """Upload the local table to the analyst once; reuse stored table_id afterward."""
    tid = get_analyst_table_id()
    if tid:
        return tid
    r = await client.post(
        f"{DATA_ANALYST_URL}/tables",
        json={"records": df_records_for_analyst(table.df)},
    )
    r.raise_for_status()
    data = r.json()
    tid = data.get("table_id")
    if not tid:
        raise RuntimeError(f"analyst /tables missing table_id: {data}")
    set_analyst_table_id(tid)
    return tid


async def _analyze_with_table(message: str, table) -> tuple[str, dict]:
    """Call data analyst /analyze; return (reply_text, response_json)."""
    async with httpx.AsyncClient(timeout=180.0) as client:
        tid = await _ensure_analyst_table(client, table)
        r = await client.post(
            f"{DATA_ANALYST_URL}/analyze",
            json={"question": message, "table_id": tid},
        )
        data = r.json()
    if r.status_code >= 400 and not data.get("answer"):
        detail = data.get("detail") or data.get("error") or r.text
        return f"Analyst error: {detail}", data
    reply = data.get("answer") or data.get("detail") or data.get("error") or "No response"
    return reply, data


@rt("/chat/send", methods=["POST"])
async def chat_send(message: str, session, ui_mode: str = "chat"):
    user_msg = message.strip()
    if not user_msg:
        return ""

    mode = (ui_mode or "chat").strip().lower()
    table = get_latest_table()

    # Table / Chart mode with a loaded table → data analyst
    # Only OOB updates (table banner / chart panel) — do not append to #chat-messages.
    if mode == "table":
        if table is None or table.df.empty:
            hint = (
                "No table loaded yet. Switch to Chat, run a data query first, "
                "then come back to Table to analyze it."
            )
            return analyze_reply(hint, oob=True)
        try:
            reply, data = await _analyze_with_table(user_msg, table)
        except Exception as e:  # noqa: BLE001
            err = f"Analyst request failed: {type(e).__name__}: {e}"
            return analyze_reply(err, oob=True)

        chart_title = None
        plan_chart = data.get("plan_chart") if isinstance(data, dict) else None
        if isinstance(plan_chart, dict):
            chart_title = plan_chart.get("title")
        plotly_json = data.get("plotly_json") if isinstance(data, dict) else None
        if plotly_json:
            store_chart_from_plotly_json(plotly_json, title=chart_title or "Chart")
            reply = f"{reply}\n\nChart updated — open the Chart icon to view it."

        parts = [analyze_reply(reply, oob=True)]
        if plotly_json:
            parts.extend([chart_badge(oob=True), chart_panel(oob=True)])
        return tuple(parts)

    # Default: blue controller
    sid = session.get("session_id", "").strip()
    payload: dict = {"message": user_msg, "use_history": True}
    if sid:
        payload["session_id"] = sid

    async with httpx.AsyncClient(timeout=180.0) as client:
        r = await client.post(CONTROLLER_URL, json=payload)
        data = r.json()

    reply = data.get("response") or data.get("detail") or "No response"
    parsed = data.get("parsed")
    loaded = ingest_parsed(parsed) if isinstance(parsed, dict) else None
    if loaded:
        reply = f"{reply}\n\nLoaded {loaded.n_rows} row{'s' if loaded.n_rows != 1 else ''} into Table."

    new_sid = (data.get("session_id") or sid or "").strip()
    if new_sid:
        session["session_id"] = new_sid

    parts = [
        oob_chat_bubble("user", user_msg),
        oob_chat_bubble("assistant", reply),
        Script(
            "requestAnimationFrame(()=>{const c=document.getElementById('chat-messages');"
            "if(c)c.scrollTop=c.scrollHeight;})"
        ),
    ]
    if loaded:
        parts.extend([table_badge(oob=True), table_panel(oob=True), analyze_reply("", oob=True)])
    return tuple(parts)


if __name__ == '__main__':
    serve(appname='cole_compass.main', port=4545)
